[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-metaheuristics.ipynb)

# Optimization & Metaheuristics

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

A foundational introduction to three techniques that sit adjacent to core ML — general-purpose optimization strategies and a data-labelling paradigm. This page will grow into a fuller standalone treatment in a future update.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Simulated Annealing and Genetic Algorithms are **general-purpose optimization strategies**, not ML algorithms in the classification/regression/clustering sense — they're tools for searching a solution space when gradient-based methods (like the gradient descent used throughout this course) don't apply, because the objective is non-differentiable, discontinuous, or has a huge number of local optima. Active Learning is a genuinely different paradigm from everything else in this course — it's about *which data to label next*, not which model to fit. This page introduces all three at a foundational level; deeper treatment is planned for later.

## ⚠ Advanced: Simulated Annealing

Named after the metallurgical process of heating and slowly cooling metal to reach a low-energy crystalline state, Simulated Annealing searches a solution space by occasionally accepting *worse* solutions — especially early on — to avoid getting permanently stuck in a local optimum, the way plain gradient descent or greedy hill-climbing can.

$$P(\text{accept worse solution}) = \exp\!\left(-\dfrac{\Delta E}{T}\right) \qquad \Delta E = \text{increase in cost},\ T = \text{current "temperature"}$$

T starts high (frequently accepts worse moves — broad exploration) and is gradually lowered via a cooling schedule (T ← T·γ each iteration, γ < 1), narrowing the search until it behaves like plain greedy descent by the end — settling into a good, though not provably global, optimum.

In [ ]:
import numpy as np

# Toy delivery-route cost function with many local minima (stand-in for a real routing problem)
def route_cost(x):
    return np.sin(3*x) * np.exp(-0.1*x) + 0.05*x**2

def simulated_annealing(cost_fn, x0, T0=10.0, cooling=0.95, n_iter=300):
    x, T = x0, T0
    best_x, best_cost = x, cost_fn(x)
    for i in range(n_iter):
        x_new = x + np.random.normal(0, 1.5)         # propose a random neighbouring solution
        delta = cost_fn(x_new) - cost_fn(x)
        if delta < 0 or np.random.rand() < np.exp(-delta/T):  # accept if better, or probabilistically if worse
            x = x_new
            if cost_fn(x) < best_cost:
                best_x, best_cost = x, cost_fn(x)
        T *= cooling                                       # cool down — accept fewer bad moves over time
    return best_x, best_cost

np.random.seed(7)
best_x, best_cost = simulated_annealing(route_cost, x0=15.0)
print(f"Best solution found: x={best_x:.3f}, cost={best_cost:.3f}")
# Compare: naive greedy descent from the same start point gets trapped in the FIRST local minimum it meets

## ⚠ Advanced: Genetic Algorithms

Genetic Algorithms (GAs) search by evolving a **population** of candidate solutions across generations, borrowing the mechanics of biological evolution: selection, crossover (recombination), and mutation.

- **Initialize:** generate a random population of candidate solutions ("chromosomes")

- **Evaluate:** score every candidate with a fitness function

- **Select:** probabilistically choose fitter candidates as "parents" (e.g., roulette-wheel or tournament selection)

- **Crossover:** combine pairs of parents to produce offspring (e.g., splice two parents' encodings at a random point)

- **Mutate:** randomly perturb offspring with small probability, to maintain diversity

- Repeat for many generations — the population's average fitness tends to improve over time

In [ ]:
import numpy as np

# Optimising a delivery-hub layout: 6 binary "chromosome" positions (open=1/closed=0 hub)
def fitness(chromosome, costs, coverage):
    if chromosome.sum() == 0: return -1e9
    return coverage @ chromosome - 0.4 * (costs @ chromosome)  # maximise coverage, penalise cost

np.random.seed(3)
n_genes = 6
costs = np.random.uniform(5,20,n_genes)      # hub operating cost
coverage = np.random.uniform(10,30,n_genes)  # customers reached if this hub is open

pop = np.random.randint(0,2,(20,n_genes))        # population of 20 candidate layouts
for gen in range(50):
    scores = np.array([fitness(c, costs, coverage) for c in pop])
    parents = pop[np.argsort(scores)[-10:]]              # select top 10 (survival of the fittest)
    children = []
    for _ in range(10):
        p1, p2 = parents[np.random.randint(10,size=2)]
        cut = np.random.randint(1,n_genes)
        child = np.concatenate([p1[:cut], p2[cut:]])          # crossover
        if np.random.rand() < 0.1: child[np.random.randint(n_genes)] ^= 1  # mutation
        children.append(child)
    pop = np.vstack([parents, children])

best = pop[np.argmax([fitness(c, costs, coverage) for c in pop])]
print(f"Best hub layout found: {best}  (1=open, 0=closed)")

GAs shine on exactly this kind of problem: combinatorial, discrete choices (open/closed, included/excluded) where the search space is far too large to enumerate exhaustively, and where no clean gradient exists to follow.

### ⚠ Binary vs. Continuous (Real-Parameter) GA

The worked example above uses a **binary GA** — each gene is a 0/1 bit, natural for on/off or combinatorial decisions. When the underlying variables are genuinely continuous (e.g., optimising a set of real-valued sensor thresholds), a **continuous GA** is usually more effective: each gene stores a real number directly, crossover blends parent values (e.g., a random weighted average between two parents' genes) rather than splicing bits, and mutation adds small Gaussian noise rather than flipping a bit. This avoids the resolution/precision trade-offs binary encoding forces (more bits per gene = finer precision but slower search), at the cost of needing continuity-aware operators.

### ⚠ Selection Strategies — How Parents Are Chosen

| Method | Mechanism | Trade-off |
|---|---|---|
| **Roulette Wheel** | P(select i) = fitness(i) / Σⱼfitness(j) — each candidate gets a wheel slice proportional to its fitness | Simple, but a single dominant individual (e.g., 90% of total fitness) starves the rest of selection chances |
| **Rank Selection** | Sort by fitness, assign selection probability by *rank* (1…N) rather than raw fitness value | Prevents one outlier from dominating; slower convergence but preserves diversity |
| **Tournament Selection** | Randomly sample k individuals, the fittest of the k wins a parent slot; repeat | Tunable selection pressure via k; simple, efficient, and the most widely used in practice |
| **Boltzmann Selection** | P = exp[−(f_max − f(xᵢ))/T], with T cooling over generations — directly borrows the Simulated Annealing formula above | Selection pressure increases as T cools, mirroring annealing's explore-then-exploit schedule |

### ⚠ Crossover Variants

The worked example used single-point crossover (splice at one random cut). Two common alternatives address its main weakness — genes near each other on the chromosome always get inherited together, regardless of whether that's meaningful:

$$\text{Single-point: parent bits swap after ONE cut position} \quad\big|\quad \text{Two-point: swap the SEGMENT between two cuts} \quad\big|\quad \text{Uniform: each gene independently inherited from either parent with 50\% probability}$$

Uniform crossover explores the search space most aggressively (no positional bias at all) but can also destroy good partial solutions ("building blocks") more readily than single/two-point — another explore/exploit trade-off, this time in the crossover operator itself rather than selection.

### ⚠ Convergence Criteria & Elitism

Unlike Simulated Annealing's fixed cooling schedule, a GA needs an explicit stopping rule. Common choices: stop when the **best individual's** fitness stops improving for N generations, when the **population's fitness variance** collapses (everyone has converged to near-identical solutions), or simply after a fixed generation budget.

> **💡 Elitism — A Cheap, Standard Safeguard**
>
> Plain generational replacement (as in the worked example, where all parents are replaced by a fresh child population) can occasionally lose the best-found solution to unlucky crossover/mutation. **Elitism** — always carrying the single best individual (or top few) forward into the next generation unchanged — guarantees the algorithm's best score never gets worse across generations, at negligible extra cost. Nearly every practical GA implementation uses it by default.

### ⚠ Why Does This Work? The Schema Theorem (Brief)

Holland's **Schema Theorem** is the classic theoretical justification for GAs: it shows that short, low-order, above-average-fitness building blocks ("schemas" — partial bit patterns, e.g. `1**0*1`, where `*` means "don't care") tend to receive *exponentially increasing* representation in the population across generations. Informally, this is the theoretical grounding for the **Building Block Hypothesis**: a GA works not by getting lucky with any single full solution, but by implicitly discovering and recombining many small, good partial patterns in parallel — the reason crossover, not just mutation, is doing genuine algorithmic work rather than just adding randomness.

### ⚠ Symbolic Regression — Evolving Equations, Not Parameter Vectors

Every regression model earlier in this course fixes a functional *form* in advance — Linear Regression commits to ŷ=θ₀+θ₁x, Polynomial Regression commits to a chosen degree — and uses optimisation only to find the best *parameters* within that fixed form. **Symbolic Regression** removes that assumption entirely: it uses Genetic Programming, a variant of the GA above where each "chromosome" is an *expression tree* rather than a fixed-length bit string, to search over the space of mathematical formulas themselves — discovering the functional form and its constants simultaneously.

> **🌳 An Individual Is Now a Tree, Not a String**
>
> A candidate solution like `y = 3.2·x₁ + sin(x₂) − 1.7` is represented as a tree: a root operator node (`+`) with children (`*`, `sin(x₂)`, constant `−1.7`), and so on down to leaf nodes (input variables and constants). Genetic operators are redefined for this tree structure: **mutation** replaces a randomly chosen subtree with a new random subtree; **crossover** swaps subtrees between two parent trees; **fitness** is typically R² or negative MSE on the training data, often penalised by tree size (*parsimony pressure*) so the search doesn't drift toward needlessly bloated equations that overfit.

|  | Polynomial Regression | Symbolic Regression |
|---|---|---|
| What's searched | Coefficients only — form (degree) is chosen by the analyst beforehand | Both the form *and* the coefficients, simultaneously |
| Search method | Closed-form / gradient descent (convex, exact) | Genetic Programming (evolutionary, no convexity guarantee) |
| Output | A polynomial of the pre-chosen degree | Any expression the operator set allows — polynomial, trigonometric, rational, mixed |
| Interpretability | High — a specific coefficient vector | High — a literal human-readable equation, not a black box |
| Risk | Wrong degree chosen → underfit/overfit | Overly complex evolved expressions → overfit; no convergence guarantee |

A Mehta Textiles process engineer trying to relate loom speed and yarn tension to fabric wastage percentage, without knowing in advance whether the true relationship is linear, quadratic, or involves some interaction term, is exactly the scenario Symbolic Regression is built for — instead of guessing a polynomial degree and checking residuals, the search discovers the functional relationship directly. In practice, the standard sklearn-compatible library for this is `gplearn` (`SymbolicRegressor`), which implements exactly the tree-based Genetic Programming loop described above.

## ⚠ Advanced: Active Learning

Every algorithm in this course so far assumes you already have a labelled training set. Active Learning flips the question: given a large pool of *unlabelled* data and a limited labelling budget (human review is expensive), **which specific points should you request labels for** to improve the model fastest?

$$\text{Uncertainty Sampling: query the point } x^* = \operatorname*{argmax}_x H(y\mid x) \qquad \text{(entropy of the model's current predicted distribution)}$$

The intuition: labelling a point the model is already highly confident about teaches it almost nothing new. Labelling a point it's most uncertain about (predicted probabilities close to 0.5 for binary classification, or high entropy across classes) is expected to be maximally informative.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_pool, y_pool = X[:800], y[:800]          # pretend these are UNLABELLED (labels hidden until queried)
X_test, y_test = X[800:], y[800:]

# Start with a tiny labelled seed set, grow it via uncertainty sampling vs random sampling
def run_strategy(strategy, n_rounds=15, batch=10):
    labelled_idx = list(np.random.RandomState(0).choice(800, 20, replace=False))
    accs = []
    for _ in range(n_rounds):
        clf = LogisticRegression(max_iter=500).fit(X_pool[labelled_idx], y_pool[labelled_idx])
        accs.append(accuracy_score(y_test, clf.predict(X_test)))
        remaining = [i for i in range(800) if i not in labelled_idx]
        if strategy == 'uncertainty':
            probs = clf.predict_proba(X_pool[remaining])
            entropy = -(probs * np.log(probs + 1e-9)).sum(axis=1)
            query = [remaining[i] for i in np.argsort(entropy)[-batch:]]  # most uncertain
        else:  # random baseline
            query = list(np.random.choice(remaining, batch, replace=False))
        labelled_idx += query
    return accs

acc_uncertainty = run_strategy('uncertainty')
acc_random = run_strategy('random')
print(f"After {20+15*10} labels — Uncertainty sampling: {acc_uncertainty[-1]:.3f}   Random: {acc_random[-1]:.3f}")

For the same labelling budget (170 points), uncertainty sampling reaches meaningfully higher accuracy than randomly choosing which points to label — the core promise of Active Learning: better model quality per labelling dollar spent, which matters enormously when labels require expert time (radiologists, translators, domain specialists).

## Try It — Watch Simulated Annealing Search the Same Cost Landscape

This plots the exact `route_cost(x)` function from the code above, starting from the same x₀=15. Step through the search one move at a time (green = accepted, red = a worse move rejected) or run all 300 iterations at once, and watch the temperature cool and the search settle near a good minimum. This run uses fresh randomness each time, so the exact path won't reproduce the seeded Python output above — but it's playing by the same rules on the same landscape.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Annealing acceptance probability

Write `p_accept(delta, T)`: a move that lowers the cost (`delta <= 0`) is always accepted; a worse move (`delta > 0`) is accepted with probability `exp(-delta / T)`.

In [ ]:
import numpy as np
def p_accept(delta, T):
    pass   # TODO


In [ ]:
try:
    check("improvement always accepted", p_accept(-2.0, 1.0) == 1)
    check("worse move at high T is likely", p_accept(1.0, 100.0) > 0.98)
    check("worse move at low T is rare", p_accept(1.0, 0.1) < 1e-4)
    check("formula", abs(p_accept(2.0, 4.0) - np.exp(-0.5)) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def p_accept(delta, T):
    return 1.0 if delta <= 0 else float(np.exp(-delta / T))

```

</details>

### Exercise 2 · Medium · Genetic crossover and mutation

Write `crossover(a, b, point)` returning the child that takes `a[:point]` followed by `b[point:]` (NumPy arrays), and `mutate(chrom, idx)` returning a copy with bit `idx` flipped.

In [ ]:
import numpy as np
def crossover(a, b, point):
    pass   # TODO
def mutate(chrom, idx):
    pass   # TODO


In [ ]:
try:
    a, b = np.array([1, 1, 1, 1, 1, 1]), np.array([0, 0, 0, 0, 0, 0])
    check("crossover", crossover(a, b, 2).tolist() == [1, 1, 0, 0, 0, 0])
    check("mutation flips one bit", mutate(a, 3).tolist() == [1, 1, 1, 0, 1, 1])
    check("mutation does not modify the parent", a.tolist() == [1, 1, 1, 1, 1, 1])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def crossover(a, b, point):
    return np.concatenate([a[:point], b[point:]])
def mutate(chrom, idx):
    c = chrom.copy()
    c[idx] = 1 - c[idx]
    return c

```

</details>

### Exercise 3 · Stretch · Anneal a simple cost function

Write `anneal(f, x0, seed=0)` that runs simulated annealing (start `T=5`, multiply by 0.97 each step, 600 steps, proposal `x + N(0, 0.5)`, accept with your `p_accept`) and returns the best `x` ever seen. It should find the minimum of `(x - 3)**2` starting from -8.

In [ ]:
import numpy as np
def anneal(f, x0, seed=0):
    pass   # TODO


In [ ]:
try:
    best = anneal(lambda x: (x - 3) ** 2, -8.0)
    check("finds x near 3", abs(best - 3) < 0.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def anneal(f, x0, seed=0):
    rng = np.random.default_rng(seed)
    x, T = x0, 5.0
    best_x, best_c = x, f(x)
    for _ in range(600):
        cand = x + rng.normal(0, 0.5)
        if rng.random() < p_accept(f(cand) - f(x), T):
            x = cand
        if f(x) < best_c:
            best_x, best_c = x, f(x)
        T *= 0.97
    return best_x

```

Early on, high temperature lets the search climb out of local minima; as it cools it settles into the best valley it found.

</details>

---
*Back to the course: **Machine Learning End To End → Optimization & Metaheuristics**.*